In [144]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
import unittest

In [155]:
def loss_fn(z, Sigma_tilde):
    """
        z: vector supporting K(z)
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # defines jax-numpy loss
    d = Sigma_tilde.shape[0]
    K_z = get_K(z, d)
    log_det_term = jnp.log(jnp.linalg.det(K_z))
    trace_term = jnp.trace(jnp.matmul(K_z, Sigma_tilde))
    #print(-log_det_term + trace_term)
    return -log_det_term + trace_term

def get_K(z, d):
    """
    z: vector supporting K(z)
    d: dimension of Sigma
    """
    z_mat = z.reshape((d, d))
    
    # get lower tri part w/ the diagonal
    lower_tri_indices = np.tril_indices(d)
    z_vals = z_mat[lower_tri_indices]
    q = len(z_vals)
    
    E1 = np.zeros((d, q))
    E2 = np.zeros((d, q))
    
    for k, (i_k, j_k) in enumerate(zip(*lower_tri_indices)):
        E1[i_k, k] = 1
        E2[j_k, k] = 1

    diag_z = jnp.diag(z_vals)
    
    K = E1 @ diag_z @ E2.T + E2 @ diag_z @ E1.T
    
    return K

def initialize_z(Sigma_tilde):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # Initialize z such that later K(z) can be defined as a diagonal matrix with positive entries
    z = np.zeros(Sigma_tilde.shape)
    diag_elements = np.maximum(0.01, 1.0 / np.diag(Sigma_tilde))
    diag_elements = diag_elements / 2  # avoid counting diagonal elements twice
    
    for i in range(len(diag_elements)):
        z[i, i] = diag_elements[i]

    print("Sigma_tilde", Sigma_tilde)
    print("z", z)

    z = z.ravel()
        
    return z

def binary_search_pd_point(z, grad):
    """
        z: current z vector
        grad: current gradient of the loss function wrt z
        step_size: initial step size for the search
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """ 
    lower, upper = 0.0, 1.0
    while upper - lower > 1e-6:
        # search for mid s.t. K(z - mid*grad) is in the pos def cone
        mid = (lower + upper) / 2
        z_new = z - mid * grad
        if not is_positive_definite(get_K(z_new)):
            # not sure about this yet
            upper = mid
        else:
            lower = mid
    return z - upper * grad
    

def optimize_lbfgs(Sigma_tilde, z, max_iters=100):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
        z: vector supporting K(z)
    """

    d = Sigma_tilde.shape[0]
    
    # Linesearch, comment/uncomment the desired one
    # linesearch = optax.scale_by_backtracking_linesearch(max_backtracking_steps=15)
    linesearch = optax.scale_by_zoom_linesearch(max_linesearch_steps=15)
    
    # Optimizer
    opt = optax.chain(
        optax.lbfgs(),
        # Compare with or without linesearch by commenting this line
        linesearch,
    )
    
    # Initialize
    state = opt.init(z)
    
    # Define a wrapper for the loss function that captures Sigma_tilde
    def loss_fn_wrapper(z):
        return loss_fn(z, Sigma_tilde)
    
    # Run optimization
    for i in range(max_iters):
        v, g = jax.value_and_grad(loss_fn_wrapper)(z)
        print(f'Iteration: {i}, Value:{v:.2e}')
        u, state = opt.update(g, state, z, value=v, grad=g, value_fn=loss_fn_wrapper)
        print(f"grad: {np.sum(np.abs(g))}")
        z = optax.apply_updates(z, u)
        K_z = get_K(z, d)
        log_det_1 = log_determinant_from_cholesky(K_z)
        s, log_det_2 = jnp.linalg.slogdet(K_z)
        print("compare log det", log_det_1, log_det_2)
        
    return z


# helpers # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

def is_positive_definite(M):
    return jnp.all(jnp.linalg.eigvalsh(M) > 0)

def matrix_to_lower_triangular_dict(matrix):
    # cast z information as a mapping of lower triangular elements to their indices
    d = matrix.shape[0]  # Assuming square matrix
    lower_tri_dict = {}
    for row in range(d):
        for col in range(row + 1):
            if matrix[row, col]!=0:
                lower_tri_dict[(row, col)] = float(matrix[row, col])
    return lower_tri_dict

# tests # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

class TestAuxiliaryFunctions(unittest.TestCase):
    def test_K(self):
        d = 3
        z = np.array([[1/2., 0., 0.],
                      [0., 2/2., 0.],
                      [4., 0., 3/2.]]).flatten()
        K_result = get_K(z, d)
        K_true = np.array([[1., 0., 4.],
                           [0., 2., 0.],
                           [4., 0., 3.]])
        self.assertTrue(np.allclose(K_result, K_true, atol=1e-10))

    def test_initialize_z(self):
        Sigma_tilde = np.array([[1/1., 0., 1/4.],
                               [0., 1/2., 0.],
                               [1/4., 0., 1/3.]])
        d = Sigma_tilde.shape[0]
        
        z_mat = np.array([[1/2., 0., 0.],
                          [0., 2/2., 0.],
                          [4., 0., 3/2.]])  # in dict form, z_dict = {(0,0): 1/2, (1,1): 2/2, (2,2): 3/2, (2,0): 4}
        z_init = initialize_z(Sigma_tilde)
        z_init_mat = z_init.reshape((d, d))

        for i in range(d):
            self.assertAlmostEqual(z_init_mat[i,i], z_mat[i,i], places=10)

    def test_lower_triangular_dict(self):
        
        matrix = np.array([
            [1., 0., 4.],
            [0., 2., 0.],
            [4., 0., 3.]
        ])

        expected_dict = {(0, 0): 1.0, (1, 1): 2.0, (2, 0): 4.0, (2, 2): 3.0}

        result_dict = matrix_to_lower_triangular_dict(matrix)

        self.assertEqual(result_dict, expected_dict, "The resulting dictionary does not match the expected dictionary.")

In [156]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_K (__main__.TestAuxiliaryFunctions) ... ok
test_initialize_z (__main__.TestAuxiliaryFunctions) ... ok
test_lower_triangular_dict (__main__.TestAuxiliaryFunctions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.011s

OK


Sigma_tilde [[1.         0.         0.25      ]
 [0.         0.5        0.        ]
 [0.25       0.         0.33333333]]
z [[0.5 0.  0. ]
 [0.  1.  0. ]
 [0.  0.  1.5]]


In [ ]:
Sigma_tilde = np.array([[1/1., 0., 1/4.],
                               [0., 1/2., 0.],
                               [1/4., 0., 1/3.]])
z_init = initialize_z(Sigma_tilde)
optimize_lbfgs(Sigma_tilde, z_init, max_iters=30)

Sigma_tilde [[1.         0.         0.25      ]
 [0.         0.5        0.        ]
 [0.25       0.         0.33333333]]
z [[0.5 0.  0. ]
 [0.  1.  0. ]
 [0.  0.  1.5]]
Iteration: 0, Value:1.21e+00


/Users/ceciliaferrando/Documents/UMASS/RESEARCH/GaussianPGM/privateggm/lib/python3.10/site-packages/optax/_src/transform.py:1412: UserWarning: Explicitly requested dtype float64 requested in zeros is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  lambda leaf: jnp.zeros((memory_size,) + leaf.shape, dtype=leaf.dtype),


grad: 0.5
compare log det 1.7047482 1.7047482
Iteration: 1, Value:1.05e+00
grad: 0.37878793478012085
compare log det 1.9223552 1.9223553
Iteration: 2, Value:1.03e+00
grad: 0.37658220529556274
compare log det 1.8857608 1.8857609
Iteration: 3, Value:1.02e+00
grad: 0.2437126636505127
compare log det 1.877912 1.8779123
Iteration: 4, Value:1.01e+00
grad: 0.15277568995952606
compare log det 1.8903143 1.8903145
Iteration: 5, Value:1.01e+00
grad: 0.08174976706504822
compare log det 1.9205339 1.920534
Iteration: 6, Value:1.00e+00
grad: 0.0672118067741394
compare log det 1.9539251 1.9539253
Iteration: 7, Value:1.00e+00
grad: 0.037192851305007935
compare log det 1.9736011 1.9736012
Iteration: 8, Value:1.00e+00
grad: 0.020957082509994507
compare log det 1.9850881 1.9850881
Iteration: 9, Value:1.00e+00
grad: 0.011541932821273804
compare log det 1.99156 1.9915601
Iteration: 10, Value:1.00e+00
grad: 0.006298065185546875
compare log det 1.9951375 1.9951375
Iteration: 11, Value:1.00e+00
grad: 0.0034159

In [ ]:
# functions to check that a matrix is PSD

@jax.jit
def log_determinant_from_cholesky(A):
    # Compute the Cholesky decomposition
    L = jnp.linalg.cholesky(A)

    # Check if any entry of the Cholesky factor is NaN
    is_nan = jnp.any(jnp.isnan(L))

    def nan_case(_):
        return -jnp.inf

    def valid_case(L):
        diag_L = jnp.diag(L)
        log_det = 2 * jnp.sum(jnp.log(diag_L))
        return log_det

    # Use jax.cond to choose between the cases
    return jax.lax.cond(is_nan, nan_case, valid_case, L)


# Example usage:
key = jax.random.PRNGKey(11)
A = jax.random.normal(key, (3, 3))
A = jnp.dot(A, A.T)  # Ensure A is positive semi-definite
log_det1 = log_determinant_from_cholesky(A)


# Compare to reference
s, log_det2 = jnp.linalg.slogdet(A)
print("Log-determinant:", log_det1)
print("Reference:", log_det2)